# OSAI Project 1 — Colab v3 (MobileNetV3-Large + LR-ASPP)

**목적**: FLOPs ≤ 15 GFLOPs (S_FLOPs=5) 달성 + mIoU ≥ 0.70 (S_mIoU=4) 또는 ≥ 0.75 (S_mIoU=5).

**모델**: MobileNetV3-Large (TorchVision IMAGENET1K_V2 pretrained) + LR-ASPP head (mid=64), OS=16.
- ONNX 측정: **14.89 GFLOPs** @ (1, 3, 480, 640) → S_FLOPs=5 ✓
- params: 3.10M (vs ResNet-50 v2.final 45M)

**학습**:
- Stage 1 (160K iter, COCO+VOC vanilla)
- Stage 2 (8K iter, VOC + Copy-Paste + Boundary-weighted CE)

**A100 권장** (Pro+ compute units 사용). ETA ~3h Stage 1 + ~30분 Stage 2.

필수 사전 준비:
1. Drive: `MyDrive/osai/p1/submit/img/` (1000장 jpg)
2. WandB API key를 Colab Secret `WANDB_API_KEY`로 저장
3. **Runtime → A100 + High-RAM** 선택

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 설정 변수

In [ ]:
REPO_URL    = "https://github.com/geniemo/osai.git"
BRANCH      = "improve"
CONFIG_S1   = "src/config/colab_mn3_s1.yaml"
CONFIG_S2   = "src/config/colab_mn3_s2.yaml"
DRIVE       = "/content/drive/MyDrive/osai/p1"
CKPT_DIR    = f"{DRIVE}/checkpoints_mn3"
CKPT_S2_BEST = f"{CKPT_DIR}/s2/best.pth"

import os
os.makedirs(f"{CKPT_DIR}/s1", exist_ok=True)
os.makedirs(f"{CKPT_DIR}/s2", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"S1 config: {CONFIG_S1}")
print(f"S2 config: {CONFIG_S2}")
print(f"Ckpt dir: {CKPT_DIR}")
assert os.path.exists(f"{DRIVE}/submit/img"), f"submit/img not found at {DRIVE}/submit/img"
n_imgs = len([f for f in os.listdir(f"{DRIVE}/submit/img") if f.endswith('.jpg')])
print(f"submit/img: {n_imgs} jpg files")

## 2. 저장소 clone

In [ ]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

## 3. uv 설치 + 의존성 sync

In [ ]:
!pip install -q uv
!uv sync

## 4. GPU 확인 (A100 권장)

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 5. WandB 로그인

In [ ]:
from google.colab import userdata
import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!uv run wandb login

## 6. Test images Colab 로컬로 복사

In [ ]:
!mkdir -p submit/img
!cp -r {DRIVE}/submit/img/. submit/img/
!ls submit/img | wc -l

## 7. 데이터 다운로드 (VOC + COCO) — 이미 있으면 빠름

In [ ]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

## 8. COCO mask cache 사전 생성

이전 학습에서 캐시 만들었으면 1초 안에 끝남.

In [ ]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

## 9. 모델 + FLOPs 사전 검증 (학습 시작 전 필수)

ONNX 측정으로 FLOPs ≤ 15 G 확인. 만약 초과하면 학습 중단.

In [ ]:
# Stage 2 config 기준 모델 빌드 → ONNX export → FLOPs 측정
!uv run python -c "\
import torch, yaml; \
from src.models.builder import build_model; \
cfg = yaml.safe_load(open('src/config/colab_mn3_s2.yaml')); \
m = build_model(cfg); m.eval(); m.export_mode(); \
torch.onnx.export(m, torch.zeros(1,3,480,640), '/tmp/check.onnx', opset_version=18, do_constant_folding=True); \
from src.utils.flops import count_onnx_flops; \
mac, _ = count_onnx_flops('/tmp/check.onnx', (1,3,480,640)); \
print(f'GFLOPs: {mac*2/1e9:.3f} (need <=15 for S_FLOPs=5)'); \
print(f'Params: {sum(p.numel() for p in m.parameters())/1e6:.2f}M'); \
assert mac*2/1e9 <= 15.0, 'FLOPs over 15 — abort training'"

## 10-1. Stage 1 학습 (vanilla, 160K iter, ~3h on A100)

COCO+VOC mixed, no Copy-Paste, no Boundary loss. ckpt를 Drive에 저장 (resume 지원).

In [ ]:
!uv run python -m src.train --config {CONFIG_S1} --stage 1

## 10-2. Stage 2 학습 (Copy-Paste + Boundary, 8K iter, ~30분)

Stage 1 best ckpt에서 bootstrap. VOC train만 사용.

In [ ]:
!uv run python -m src.train --config {CONFIG_S2} --stage 2

## 11. Validation mIoU (raw + TTA)

TTA: scales [0.5, 0.75, 1.0, 1.25, 1.5] + hflip

In [ ]:
!uv run python -m src.eval --config {CONFIG_S2} --ckpt {CKPT_S2_BEST}
print('---')
!uv run python -m src.eval_tta --config {CONFIG_S2} --ckpt {CKPT_S2_BEST}

## 12. 추론 (TTA) — submit/img → submit/pred

In [ ]:
!mkdir -p submit/pred
!uv run python -m src.infer \
    --config {CONFIG_S2} \
    --ckpt {CKPT_S2_BEST} \
    --input submit/img \
    --output submit/pred

## 13. ONNX export (채점 사이트용)

In [ ]:
!uv run python -m src.export_onnx \
    --config {CONFIG_S2} \
    --ckpt {CKPT_S2_BEST} \
    --out {DRIVE}/model_structure_mn3.onnx

## 14. FLOPs 최종 측정

In [ ]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure_mn3.onnx

## 15. PNG zip 생성 (채점 사이트 제출용)

In [ ]:
!uv run python -m src.package_submission \
    --pred submit/pred \
    --out {DRIVE}/submission_pred_mn3.zip

## 16. 결과물 확인

In [ ]:
!ls -la {DRIVE}/
!ls -la {CKPT_DIR}/s1/ {CKPT_DIR}/s2/